# Inference Notebook — Imane (GNN + Classical ML)

Load pre-trained models and run predictions without retraining.

**Models available:**
- 11 classical ML models (top 3: LightGBM, GradientBoosting, XGBoost)
- 3 GNN models: SimpleGNN, CGCNN, LSTM

## 1. Imports

In [ ]:
import pandas as pd, numpy as np, pickle, os, warnings
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator, Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool, global_add_pool
from torch_geometric.data import Data, Batch
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings('ignore')

# Detect device
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Load Data
Same preprocessed dataset used for training.

In [ ]:
df = pd.read_csv('data/curated_data_rdkit_Imane.csv')
df.rename(columns={'logC50':'log_LC50','SMILES_curated':'SMILES'}, inplace=True)
meta = df['CAS'].astype(str).str.contains('Meta', na=False)
df = df[~meta].reset_index(drop=True)
df['mol'] = df['SMILES'].apply(lambda s: Chem.MolFromSmiles(s) if s else None)
df = df.dropna(subset=['mol']).reset_index(drop=True)
df['toxic'] = (df['log_LC50'] > 1.0).astype(int)
print(f'Molecules: {len(df)} | Toxic: {df["toxic"].sum()} ({df["toxic"].mean()*100:.1f}%)')
df.head(3)

## 3. Load Saved Classical ML Models

In [ ]:
with open('models/classical_models_Imane.pkl', 'rb') as f:
    cp = pickle.load(f)

trained_models = cp['models']
top3 = cp['top3']
sel = cp['preprocessor']['sel']
scaler = cp['preprocessor']['scaler']
feat_names = cp['preprocessor']['feat_names']
needs_scaling = cp['preprocessor']['needs_scaling']

print('Loaded classical models:', list(trained_models.keys()))
print(f'Top 3: {top3}')
print(f'Features: {len(feat_names)}')

### 3.1 Reproduce Test Set Results

In [ ]:
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score)

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = np.array([gen.GetFingerprint(m) for m in df['mol']])
desc_names = [d[0] for d in Descriptors._descList]
calculator = MoleculeDescriptors.MolecularDescriptorCalculator(desc_names)
descs = np.array([list(calculator.CalcDescriptors(m)) for m in df['mol']])
X_raw = np.hstack([fps, descs])
y = df['toxic'].values
X = sel.transform(X_raw)

from sklearn.model_selection import train_test_split
ix_train, ix_test, y_train, y_test = train_test_split(
    np.arange(len(y)), y, test_size=0.2, random_state=42, stratify=y)
X_train, X_test = X[ix_train], X[ix_test]
X_test_s = scaler.transform(X_test)
print(f'Test set: {len(X_test)} molecules')

results = []
for name, m in trained_models.items():
    use_s = name in needs_scaling
    X_te = X_test_s if use_s else X_test
    if hasattr(m, 'predict_proba'):
        proba = m.predict_proba(X_te)[:, 1]
    else:
        proba = m.decision_function(X_te)
    pred = (proba > 0.5).astype(int)
    results.append({'model': name,
        'auc_roc': roc_auc_score(y_test, proba),
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'specificity': recall_score(1-y_test, 1-pred, zero_division=0)})

dfr = pd.DataFrame(results).sort_values('auc_roc', ascending=False).reset_index(drop=True)
print('=== Classical Benchmark Results ===')
print(dfr[['model','auc_roc','accuracy','f1','precision','recall','specificity']].to_string(index=False))

### 3.2 Predict on New SMILES (Classical Models)

In [ ]:
def predict_classical(smiles_list, model_name=None):
    """
    Predict toxicity for new SMILES using classical models.
    
    Parameters:
    - smiles_list: list of SMILES strings
    - model_name: which model to use (None = all top 3)
    
    Returns DataFrame with predictions.
    """
    mols = [Chem.MolFromSmiles(s) for s in smiles_list]
    valid = [m is not None for m in mols]
    if not all(valid):
        print(f'Warning: {sum(1 for v in valid if not v)} invalid SMILES')
    
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    fps = np.array([gen.GetFingerprint(m) for m in mols if m is not None])
    calculator = MoleculeDescriptors.MolecularDescriptorCalculator(desc_names)
    descs = np.array([list(calculator.CalcDescriptors(m)) for m in mols if m is not None])
    X_new = sel.transform(np.hstack([fps, descs]))
    
    models_to_use = [model_name] if model_name else top3
    rows = []
    for name in models_to_use:
        if name not in trained_models:
            print(f'Unknown model: {name}')
            continue
        m = trained_models[name]
        use_s = name in needs_scaling
        X_te = scaler.transform(X_new) if use_s else X_new
        proba = m.predict_proba(X_te)[:, 1]
        for i, smi in enumerate([s for s, v in zip(smiles_list, valid) if v]):
            rows.append({'SMILES': smi, 'Model': name,
                         'Toxicity_Probability': round(proba[i], 4),
                         'Prediction': 'Toxic' if proba[i] > 0.5 else 'Non-toxic'})
    return pd.DataFrame(rows)

# Example: predict on a few molecules
example_smiles = ['CCO', 'c1ccccc1', 'O=C(O)c1ccccc1', 'CCCCCCCCBr']
predict_classical(example_smiles)

---
## 4. GNN Models

Define model architectures and load trained state dicts.

### 4.1 Featurization Functions

In [ ]:
ATOM_DIM = 60; MAX_DEG = 6; MAX_HS = 5
HYBRID_MAP = {Chem.HybridizationType.SP:0, Chem.HybridizationType.SP2:1,
              Chem.HybridizationType.SP3:2, Chem.HybridizationType.SP3D:3,
              Chem.HybridizationType.SP3D2:4}
CHIRAL_MAP = {Chem.ChiralType.CHI_TETRAHEDRAL_CW:0,
              Chem.ChiralType.CHI_TETRAHEDRAL_CCW:1,
              Chem.ChiralType.CHI_UNSPECIFIED:2}
BOND_MAP = {Chem.BondType.SINGLE:0, Chem.BondType.DOUBLE:1,
            Chem.BondType.TRIPLE:2, Chem.BondType.AROMATIC:3}

def atom_feats(a):
    o = [0]*ATOM_DIM; an = a.GetAtomicNum()
    if an < ATOM_DIM: o[an] = 1
    return torch.tensor(o + [min(a.GetDegree(),MAX_DEG)/MAX_DEG,
        np.clip(a.GetFormalCharge(),-3,3)/3,
        min(a.GetTotalNumHs(),MAX_HS)/MAX_HS,
        HYBRID_MAP.get(a.GetHybridization(),0)/4,
        1.0 if a.GetIsAromatic() else 0.0,
        CHIRAL_MAP.get(a.GetChiralTag(),0)/2,
        (a.GetMass()-12)/100], dtype=torch.float)

def bond_feats(b):
    return torch.tensor([BOND_MAP.get(b.GetBondType(),0)/3,
        1.0 if b.GetIsConjugated() else 0.0,
        1.0 if b.IsInRing() else 0.0], dtype=torch.float)

def mol_to_graph(mol):
    atoms = list(mol.GetAtoms())
    x = torch.stack([atom_feats(a) for a in atoms])
    ei, ea = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        e = bond_feats(b)
        ei.extend([[i,j],[j,i]]); ea.extend([e,e])
    ei = torch.tensor(ei, dtype=torch.long).T if ei else torch.empty((2,0), dtype=torch.long)
    ea = torch.stack(ea) if ea else torch.empty((0,3), dtype=torch.float)
    return Data(x=x, edge_index=ei, edge_attr=ea)

### 4.2 Model Architecture Definitions

In [ ]:
class SimpleGNN(nn.Module):
    def __init__(self, nd, hd=128, nl=3, dp=0.2):
        super().__init__()
        self.np = nn.Linear(nd, hd)
        self.convs = nn.ModuleList([GCNConv(hd, hd) for _ in range(nl)])
        self.norms = nn.ModuleList([nn.BatchNorm1d(hd) for _ in range(nl)])
        self.fc = nn.Sequential(nn.Linear(hd, 64), nn.GELU(), nn.Dropout(dp),
                                 nn.Linear(64, 32), nn.GELU(), nn.Dropout(dp), nn.Linear(32, 1))
        self.dp = nn.Dropout(dp)
    def forward(self, d):
        x, ei, b = d.x, d.edge_index, d.batch
        x = self.np(x)
        for c, no in zip(self.convs, self.norms):
            r = x; x = c(x, ei); x = no(x + r); x = F.gelu(x); x = self.dp(x)
        x = global_mean_pool(x, b)
        return self.fc(x).squeeze(-1)

class CGCNNConv(nn.Module):
    def __init__(self, nd, ed):
        super().__init__()
        self.lin_z = nn.Linear(2 * nd + ed, nd)
        self.lin_g = nn.Linear(2 * nd + ed, nd)
        self.lin_x = nn.Linear(nd, nd)
    def forward(self, x, ei, ea):
        i, j = ei[0], ei[1]
        z_in = torch.cat([x[i], x[j], ea], dim=-1)
        z = self.lin_z(z_in)
        g = torch.sigmoid(self.lin_g(z_in))
        msg = g * self.lin_x(z)
        out = global_add_pool(msg, i, x.shape[0])
        return F.gelu(x + out)

class CGCNN(nn.Module):
    def __init__(self, nd, ed=3, hd=128, nl=4, dp=0.2):
        super().__init__()
        self.np = nn.Linear(nd, hd)
        self.convs = nn.ModuleList([CGCNNConv(hd, ed) for _ in range(nl)])
        self.fc = nn.Sequential(nn.Linear(hd, 64), nn.GELU(), nn.Dropout(dp),
                                 nn.Linear(64, 32), nn.GELU(), nn.Dropout(dp), nn.Linear(32, 1))
        self.dp = nn.Dropout(dp)
    def forward(self, d):
        x, ei, ea, b = d.x, d.edge_index, d.edge_attr, d.batch
        x = self.np(x)
        for c in self.convs: x = c(x, ei, ea); x = self.dp(x)
        x = global_mean_pool(x, b)
        return self.fc(x).squeeze(-1)

In [ ]:
SMILES_CHARS = 'CAM NnPpSsiI1234567890BDFGcbrfl-+=#()[]/\\@.%'
char2idx = {c: i+2 for i, c in enumerate(SMILES_CHARS)}
char2idx['<PAD>'] = 0; char2idx['<UNK>'] = 1

def tokenize(smiles, max_len=256):
    ids = []
    for s in smiles:
        toks = []; i = 0
        while i < len(s):
            if i+1 < len(s) and s[i:i+2] in ('Cl','Br','Si','Se','As'):
                toks.append(s[i:i+2]); i+=2
            elif s[i] in char2idx: toks.append(s[i]); i+=1
            else: toks.append('<UNK>'); i+=1
        ids.append([char2idx.get(t, 1) for t in toks[:max_len]] + [0]*(max_len - len(toks)))
    return torch.tensor(ids, dtype=torch.long)

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size=len(char2idx), emb_dim=128, hd=256, nl=3, dp=0.3):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hd, num_layers=nl, batch_first=True, bidirectional=True, dropout=dp)
        self.fc = nn.Sequential(nn.Linear(hd*2, 128), nn.GELU(), nn.Dropout(dp),
                                 nn.Linear(128, 64), nn.GELU(), nn.Dropout(dp), nn.Linear(64, 1))
    def forward(self, x):
        x = self.emb(x)
        x, (h, c) = self.lstm(x)
        x = torch.cat([h[-2], h[-1]], dim=1)
        return self.fc(x).squeeze(-1)

### 4.3 Load Trained GNN Weights (.pth files)

In [ ]:
gnn_models = {}

# SimpleGNN
model = SimpleGNN(nd=67).to(device)
model.load_state_dict(torch.load('models/SimpleGNN_binary.pth', map_location=device, weights_only=True))
model.eval()
gnn_models['SimpleGNN'] = model
print('Loaded SimpleGNN')

# CGCNN
model = CGCNN(nd=67).to(device)
model.load_state_dict(torch.load('models/CGCNN_binary.pth', map_location=device, weights_only=True))
model.eval()
gnn_models['CGCNN'] = model
print('Loaded CGCNN')

# LSTM
model = LSTMClassifier().to(device)
model.load_state_dict(torch.load('models/LSTM_binary.pth', map_location=device, weights_only=True))
model.eval()
gnn_models['LSTM'] = model
print('Loaded LSTM')

### 4.4 Reproduce GNN Test Set Results

In [ ]:
# Prepare graph data for test set
test_mols = df.iloc[ix_test]['mol'].tolist()
test_graphs = [mol_to_graph(m) for m in test_mols]
test_smiles = df.iloc[ix_test]['SMILES'].tolist()
y_test_tensor = torch.tensor(y_test.values if hasattr(y_test, 'values') else y_test, dtype=torch.float)

class GraphDataset(Dataset):
    def __init__(self, gs, y): self.gs = gs; self.y = y
    def __len__(self): return len(self.gs)
    def __getitem__(self, i): return self.gs[i], self.y[i]

def collate_graphs(b):
    gs, ts = zip(*b); return Batch.from_data_list(list(gs)), torch.stack(ts)

test_loader = DataLoader(GraphDataset(test_graphs, y_test_tensor), 32, shuffle=False, collate_fn=collate_graphs)

# LSTM loader
class LSTMDataset(Dataset):
    def __init__(self, sm, y): self.x = tokenize(sm); self.y = y
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x[i], self.y[i]

lstm_test_loader = DataLoader(LSTMDataset(test_smiles, y_test_tensor), 32, shuffle=False)

from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

def evaluate_gnn(model, loader, is_lstm=False):
    preds, targets = [], []
    model.eval()
    with torch.no_grad():
        for b in loader:
            if is_lstm:
                x, y = b[0].to(device), b[1].unsqueeze(-1)
            else:
                x, y = b[0].to(device), b[1]
            logits = model(x)
            logits_ = logits.unsqueeze(-1) if logits.dim() == 1 else logits
            preds.extend(torch.sigmoid(logits_).squeeze(-1).cpu().numpy())
            targets.extend(y.cpu().numpy())
    preds = np.array(preds); targets = np.array(targets)
    binary = (preds > 0.5).astype(int)
    return {'accuracy': accuracy_score(targets, binary),
            'auc_roc': roc_auc_score(targets, preds),
            'sensitivity': recall_score(targets, binary, zero_division=0),
            'specificity': recall_score(1-targets, 1-binary, zero_division=0),
            'precision': precision_score(targets, binary, zero_division=0),
            'f1': f1_score(targets, binary, zero_division=0)}

gnn_results = {}
for name, model in gnn_models.items():
    is_lstm = name == 'LSTM'
    loader = lstm_test_loader if is_lstm else test_loader
    gnn_results[name] = evaluate_gnn(model, loader, is_lstm)
    print(f'{name:12s}: AUC={gnn_results[name]["auc_roc"]:.4f}  '
          f'Acc={gnn_results[name]["accuracy"]:.4f}  F1={gnn_results[name]["f1"]:.4f}')

        # Combined table with classical top 3
rows = []
for n in top3:
    r = [x for x in results if x['model'] == n][0]
    rows.append({'model': f'Classical: {n}',
        'auc_roc': r.get('auc_roc'), 'accuracy': r.get('accuracy'),
        'precision': r.get('precision'), 'recall': r.get('recall'),
        'specificity': r.get('specificity'), 'f1': r.get('f1')})
for n, r in gnn_results.items():
    rows.append({'model': n, **r})
pdf = pd.DataFrame(rows).sort_values('auc_roc', ascending=False).reset_index(drop=True)
print('\n=== GNN vs Classical Comparison ===')
print(pdf.to_string(index=False))

### 4.5 Predict on New SMILES (GNN Models)

In [ ]:
def predict_gnn(smiles_list, model_name='CGCNN'):
    """Predict toxicity using a loaded GNN model."""
    if model_name not in gnn_models:
        print(f'Available: {list(gnn_models.keys())}')
        return
    model = gnn_models[model_name]
    
    mols = [Chem.MolFromSmiles(s) for s in smiles_list]
    valid = [m is not None for m in mols]
    
    if model_name == 'LSTM':
        x = tokenize([s for s, v in zip(smiles_list, valid) if v]).to(device)
        with torch.no_grad():
            probs = torch.sigmoid(model(x)).cpu().numpy()
    else:
        graphs = Batch.from_data_list([mol_to_graph(m) for m in mols if m is not None])
        graphs = graphs.to(device)
        with torch.no_grad():
            probs = torch.sigmoid(model(graphs)).cpu().numpy()
    
    results = []
    idx = 0
    for i, (smi, v) in enumerate(zip(smiles_list, valid)):
        if v:
            results.append({'SMILES': smi, 'Model': model_name,
                'Toxicity_Probability': round(float(probs[idx]), 4),
                'Prediction': 'Toxic' if probs[idx] > 0.5 else 'Non-toxic'})
            idx += 1
        else:
            results.append({'SMILES': smi, 'Model': model_name,
                'Toxicity_Probability': None, 'Prediction': 'Invalid SMILES'})
    return pd.DataFrame(results)

# Example
predict_gnn(['CCO', 'c1ccccc1', 'O=C(O)c1ccccc1', 'CCCCCCCCBr'], 'CGCNN')

---
## 5. Saved Results (for Thesis)

All performance metrics saved as CSV files in `data/`.

In [ ]:
        print('Classical benchmark:')
print(pd.read_csv('data/classical_benchmark.csv').to_string(index=False))
print('\nGNN comparison:')
print(pd.read_csv('data/gnn_comparison.csv').to_string(index=False))